# Config Fine Tuning Notebook
**This notebook will be used to explore tuning grid for BUDDY and ELPHEA**

In [1]:
import sys
from pathlib import Path
from itertools import product

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd
import torch
from torch import optim

from src.utils.helpers import set_seed, get_device
from src.data_processing.load_data import get_data_object
from src.data_processing.preprocess import prepare_link_prediction_data
from src.utils.buddy_helpers import build_buddy_cache
from src.models.elph import ELPHEdgeAware
from src.models.buddy import BUDDY
from src.models.train import fit, fit_buddy
from src.evaluation.evaluate import evaluate_split, evaluate_split_buddy

In [2]:
fixed_structural_config = {
    "num_hops": 2,
    "minhash_num_perm": 128,
    "hll_p": 8,
    "feature_propagation": "mean",
}

fixed_training_config = {
    "epochs": 15,
    "patience": 5,
    "weight_decay": 1e-4,
}

print(fixed_structural_config)
print(fixed_training_config)

{'num_hops': 2, 'minhash_num_perm': 128, 'hll_p': 8, 'feature_propagation': 'mean'}
{'epochs': 15, 'patience': 5, 'weight_decay': 0.0001}


### Cora

#### Stage1

In [3]:
set_seed(42)
device = get_device()

dataset_name = "Cora"

data = get_data_object(dataset_name, root=project_root / "data")
train_data, val_data, test_data = prepare_link_prediction_data(dataset_name, data)

print("Device:", device)
print("Dataset:", dataset_name)
print(train_data)
print(val_data)
print(test_data)

Device: cuda
Dataset: Cora
Data(x=[2708, 1433], edge_index=[2, 7392], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708], edge_label=[7392], edge_label_index=[2, 7392])
Data(x=[2708, 1433], edge_index=[2, 7392], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708], edge_label=[1054], edge_label_index=[2, 1054])
Data(x=[2708, 1433], edge_index=[2, 8446], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708], edge_label=[2110], edge_label_index=[2, 2110])


In [4]:
def run_buddy_trial(
    dataset_name: str,
    train_data_local,
    val_data_local,
    test_data_local,
    buddy_cache_local,
    trial_name: str,
    predictor_hidden_channels: int,
    dropout: float,
    lr: float,
    seed: int = 42,
    epochs: int | None = None,
    patience: int | None = None,
    monitor_hits_k: int = 100,
) -> dict:
    set_seed(seed)

    model = BUDDY(
        node_feature_dim=train_data_local.x.size(1),
        num_hops=2,
        predictor_hidden_channels=predictor_hidden_channels,
        dropout=dropout,
        structural_use_log=True,
    ).to(device)

    optimizer = optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=1e-4,
    )

    history = fit_buddy(
        model=model,
        optimizer=optimizer,
        train_data=train_data_local,
        val_data=val_data_local,
        buddy_cache=buddy_cache_local,
        device=device,
        epochs=epochs if epochs is not None else 25,
        patience=patience if patience is not None else 8,
        verbose=False,
        checkpoint_path=None,
        restore_best_model=True,
        monitor="val_hits@K",
        monitor_hits_k=monitor_hits_k,
    )

    train_metrics = evaluate_split_buddy(
        model, train_data_local, buddy_cache_local, device, hits_ks=[monitor_hits_k]
    )
    val_metrics = evaluate_split_buddy(
        model, val_data_local, buddy_cache_local, device, hits_ks=[monitor_hits_k]
    )
    test_metrics = evaluate_split_buddy(
        model, test_data_local, buddy_cache_local, device, hits_ks=[monitor_hits_k]
    )

    return {
        "dataset": dataset_name,
        "model": "BUDDY_log",
        "trial_name": trial_name,
        "seed": seed,
        "predictor_hidden_channels": predictor_hidden_channels,
        "dropout": dropout,
        "lr": lr,
        "best_epoch": history["best_epoch"],
        "best_val_loss": history["best_val_loss"],
        "best_monitor_value": history["best_monitor_value"],
        "train_auc": train_metrics["auc"],
        "train_ap": train_metrics["ap"],
        f"train_hits@{monitor_hits_k}": train_metrics[f"hits@{monitor_hits_k}"],
        "val_auc": val_metrics["auc"],
        "val_ap": val_metrics["ap"],
        f"val_hits@{monitor_hits_k}": val_metrics[f"hits@{monitor_hits_k}"],
        "test_auc": test_metrics["auc"],
        "test_ap": test_metrics["ap"],
        f"test_hits@{monitor_hits_k}": test_metrics[f"hits@{monitor_hits_k}"],
    }


def run_elphedge_trial(
    dataset_name: str,
    train_data_local,
    val_data_local,
    test_data_local,
    trial_name: str,
    hidden_channels: int,
    predictor_hidden_channels: int,
    message_hidden_channels: int,
    update_hidden_channels: int,
    dropout: float,
    lr: float,
    seed: int = 42,
    epochs: int | None = None,
    patience: int | None = None,
    monitor_hits_k: int = 100,
) -> dict:
    set_seed(seed)

    model = ELPHEdgeAware(
        in_channels=train_data_local.x.size(1),
        hidden_channels=hidden_channels,
        predictor_hidden_channels=predictor_hidden_channels,
        num_hops=2,
        minhash_num_perm=128,
        hll_p=8,
        message_hidden_channels=message_hidden_channels,
        update_hidden_channels=update_hidden_channels,
        dropout=dropout,
        use_log_features=True,
    ).to(device)

    optimizer = optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=1e-4,
    )

    history = fit(
        model=model,
        optimizer=optimizer,
        train_data=train_data_local,
        val_data=val_data_local,
        device=device,
        epochs=epochs if epochs is not None else 25,
        patience=patience if patience is not None else 8,
        verbose=False,
        checkpoint_path=None,
        restore_best_model=True,
        monitor="val_hits@K",
        monitor_hits_k=monitor_hits_k,
    )

    train_metrics = evaluate_split(
        model, train_data_local, device, hits_ks=[monitor_hits_k]
    )
    val_metrics = evaluate_split(
        model, val_data_local, device, hits_ks=[monitor_hits_k]
    )
    test_metrics = evaluate_split(
        model, test_data_local, device, hits_ks=[monitor_hits_k]
    )

    return {
        "dataset": dataset_name,
        "model": "ELPHEdgeAware_log",
        "trial_name": trial_name,
        "seed": seed,
        "hidden_channels": hidden_channels,
        "predictor_hidden_channels": predictor_hidden_channels,
        "message_hidden_channels": message_hidden_channels,
        "update_hidden_channels": update_hidden_channels,
        "dropout": dropout,
        "lr": lr,
        "best_epoch": history["best_epoch"],
        "best_val_loss": history["best_val_loss"],
        "best_monitor_value": history["best_monitor_value"],
        "train_auc": train_metrics["auc"],
        "train_ap": train_metrics["ap"],
        f"train_hits@{monitor_hits_k}": train_metrics[f"hits@{monitor_hits_k}"],
        "val_auc": val_metrics["auc"],
        "val_ap": val_metrics["ap"],
        f"val_hits@{monitor_hits_k}": val_metrics[f"hits@{monitor_hits_k}"],
        "test_auc": test_metrics["auc"],
        "test_ap": test_metrics["ap"],
        f"test_hits@{monitor_hits_k}": test_metrics[f"hits@{monitor_hits_k}"],
    }

In [5]:
buddy_cache = build_buddy_cache(
    x=train_data.x.to(device),
    edge_index=train_data.edge_index.to(device),
    num_hops=fixed_structural_config["num_hops"],
    minhash_num_perm=fixed_structural_config["minhash_num_perm"],
    hll_p=fixed_structural_config["hll_p"],
    feature_propagation=fixed_structural_config["feature_propagation"],
    cache_device=device,
)

print("BUDDY cache ready.")
print("Cache keys:", buddy_cache.keys())

BUDDY cache ready.
Cache keys: dict_keys(['num_nodes', 'num_hops', 'propagated_x_hops', 'minhash_hops', 'hll_hops', 'cardinality_hops'])


In [6]:
elphedge_trials_cora = [
    {
        "trial_name": "E1",
        "hidden_channels": 32,
        "predictor_hidden_channels": 64,
        "message_hidden_channels": 32,
        "update_hidden_channels": 32,
        "dropout": 0.2,
        "lr": 1e-2,
    },
    {
        "trial_name": "E2",
        "hidden_channels": 64,
        "predictor_hidden_channels": 64,
        "message_hidden_channels": 64,
        "update_hidden_channels": 64,
        "dropout": 0.1,
        "lr": 1e-2,
    },
    {
        "trial_name": "E3",
        "hidden_channels": 64,
        "predictor_hidden_channels": 64,
        "message_hidden_channels": 64,
        "update_hidden_channels": 64,
        "dropout": 0.2,
        "lr": 1e-2,
    },
    {
        "trial_name": "E4",
        "hidden_channels": 64,
        "predictor_hidden_channels": 128,
        "message_hidden_channels": 64,
        "update_hidden_channels": 64,
        "dropout": 0.2,
        "lr": 5e-3,
    },
    {
        "trial_name": "E5",
        "hidden_channels": 128,
        "predictor_hidden_channels": 64,
        "message_hidden_channels": 64,
        "update_hidden_channels": 64,
        "dropout": 0.2,
        "lr": 5e-3,
    },
    {
        "trial_name": "E6",
        "hidden_channels": 64,
        "predictor_hidden_channels": 64,
        "message_hidden_channels": 128,
        "update_hidden_channels": 128,
        "dropout": 0.3,
        "lr": 5e-3,
    },
]

pd.DataFrame(elphedge_trials_cora)

,trial_name,hidden_channels,predictor_hidden_channels,message_hidden_channels,update_hidden_channels,dropout,lr
0,E1,32,64,32,32,0.2,0.010
1,E2,64,64,64,64,0.1,0.010
2,E3,64,64,64,64,0.2,0.010
3,E4,64,128,64,64,0.2,0.005
4,E5,128,64,64,64,0.2,0.005
5,E6,64,64,128,128,0.3,0.005


In [7]:
elphedge_results_cora = []

for cfg in elphedge_trials_cora:
    print(f"Running Cora {cfg['trial_name']} ...")
    result = run_elphedge_trial(
        dataset_name="Cora",
        train_data_local=train_data,
        val_data_local=val_data,
        test_data_local=test_data,
        **cfg,
    )
    elphedge_results_cora.append(result)

elphedge_df_cora = pd.DataFrame(elphedge_results_cora).sort_values(
    by=["val_hits@100", "val_auc", "val_ap"],
    ascending=False,
).reset_index(drop=True)

elphedge_df_cora

Running Cora E1 ...
Running Cora E2 ...
Running Cora E3 ...
Running Cora E4 ...
Running Cora E5 ...
Running Cora E6 ...


,dataset,model,trial_name,seed,hidden_channels,predictor_hidden_channels,message_hidden_channels,update_hidden_channels,dropout,lr,...,best_val_loss,train_auc,train_ap,train_hits@100,val_auc,val_ap,val_hits@100,test_auc,test_ap,test_hits@100
0,Cora,ELPHEdgeAware_log,E1,42,32,64,32,32,0.2,0.010,...,0.596551,0.975574,0.959970,0.631764,0.840758,0.850620,0.730550,0.857617,0.860595,0.665403
1,Cora,ELPHEdgeAware_log,E4,42,64,128,64,64,0.2,0.005,...,0.608705,0.976872,0.967107,0.720509,0.836726,0.851974,0.722960,0.853418,0.861190,0.668246
2,Cora,ELPHEdgeAware_log,E6,42,64,64,128,128,0.3,0.005,...,0.604573,0.974533,0.965697,0.650974,0.832578,0.848934,0.715370,0.850537,0.856479,0.649289
3,Cora,ELPHEdgeAware_log,E3,42,64,64,64,64,0.2,0.010,...,0.523968,0.983219,0.974949,0.785985,0.823864,0.845864,0.709677,0.849828,0.864631,0.665403
4,Cora,ELPHEdgeAware_log,E2,42,64,64,64,64,0.1,0.010,...,0.547503,0.969667,0.953956,0.536255,0.812598,0.832467,0.696395,0.833140,0.837963,0.621801
5,Cora,ELPHEdgeAware_log,E5,42,128,64,64,64,0.2,0.005,...,0.615055,0.949455,0.927072,0.384470,0.820422,0.828590,0.686907,0.829400,0.825818,0.567773


In [8]:
buddy_trials_cora = [
    {"trial_name": "B1", "predictor_hidden_channels": 32,  "dropout": 0.0, "lr": 1e-2},
    {"trial_name": "B2", "predictor_hidden_channels": 64,  "dropout": 0.1, "lr": 1e-2},
    {"trial_name": "B3", "predictor_hidden_channels": 64,  "dropout": 0.2, "lr": 1e-2},
    {"trial_name": "B4", "predictor_hidden_channels": 64,  "dropout": 0.3, "lr": 5e-3},
    {"trial_name": "B5", "predictor_hidden_channels": 128, "dropout": 0.1, "lr": 5e-3},
    {"trial_name": "B6", "predictor_hidden_channels": 128, "dropout": 0.2, "lr": 5e-3},
    {"trial_name": "B7", "predictor_hidden_channels": 128, "dropout": 0.3, "lr": 1e-3},
    {"trial_name": "B8", "predictor_hidden_channels": 256, "dropout": 0.2, "lr": 1e-3},
]

pd.DataFrame(buddy_trials_cora)

,trial_name,predictor_hidden_channels,dropout,lr
0,B1,32,0.0,0.010
1,B2,64,0.1,0.010
2,B3,64,0.2,0.010
3,B4,64,0.3,0.005
4,B5,128,0.1,0.005
5,B6,128,0.2,0.005
6,B7,128,0.3,0.001
7,B8,256,0.2,0.001


In [9]:
buddy_results_cora = []

for cfg in buddy_trials_cora:
    print(f"Running Cora {cfg['trial_name']} ...")
    result = run_buddy_trial(
        dataset_name="Cora",
        train_data_local=train_data,
        val_data_local=val_data,
        test_data_local=test_data,
        buddy_cache_local=buddy_cache,
        **cfg,
    )
    buddy_results_cora.append(result)

buddy_df_cora = pd.DataFrame(buddy_results_cora).sort_values(
    by=["val_hits@100", "val_auc", "val_ap"],
    ascending=False,
).reset_index(drop=True)

buddy_df_cora

Running Cora B1 ...
Running Cora B2 ...
Running Cora B3 ...
Running Cora B4 ...
Running Cora B5 ...
Running Cora B6 ...
Running Cora B7 ...
Running Cora B8 ...


,dataset,model,trial_name,seed,predictor_hidden_channels,dropout,lr,best_epoch,best_val_loss,train_auc,train_ap,train_hits@100,val_auc,val_ap,val_hits@100,test_auc,test_ap,test_hits@100
0,Cora,BUDDY_log,B1,42,32,0.0,0.010,8,0.557614,0.992444,0.990951,0.969426,0.884205,0.891664,0.804554,0.903977,0.917355,0.735545
1,Cora,BUDDY_log,B7,42,128,0.3,0.001,11,0.586896,0.986031,0.985120,0.920455,0.861864,0.876795,0.774194,0.882228,0.902464,0.692891
2,Cora,BUDDY_log,B4,42,64,0.3,0.005,3,0.629134,0.984976,0.984307,0.918561,0.860960,0.875762,0.772296,0.880372,0.901606,0.697630
3,Cora,BUDDY_log,B5,42,128,0.1,0.005,2,0.618776,0.983840,0.982753,0.915043,0.863045,0.873251,0.768501,0.880772,0.899607,0.677725
4,Cora,BUDDY_log,B6,42,128,0.2,0.005,3,0.577548,0.985418,0.984256,0.918561,0.862537,0.876325,0.768501,0.881367,0.901750,0.690995
5,Cora,BUDDY_log,B8,42,256,0.2,0.001,7,0.591774,0.985736,0.984236,0.917208,0.861954,0.876028,0.768501,0.880961,0.902162,0.689099
6,Cora,BUDDY_log,B3,42,64,0.2,0.010,7,0.506433,0.993025,0.991401,0.969968,0.863315,0.880909,0.766603,0.891577,0.909968,0.723223
7,Cora,BUDDY_log,B2,42,64,0.1,0.010,7,0.508543,0.993223,0.991632,0.972403,0.860740,0.879937,0.766603,0.889437,0.908784,0.722275


In [10]:
elphedge_df_sorted = elphedge_df_cora.sort_values(
    by=["val_hits@100", "val_auc", "val_ap"],
    ascending=False,
).reset_index(drop=True)

elphedge_df_sorted

,dataset,model,trial_name,seed,hidden_channels,predictor_hidden_channels,message_hidden_channels,update_hidden_channels,dropout,lr,...,best_val_loss,train_auc,train_ap,train_hits@100,val_auc,val_ap,val_hits@100,test_auc,test_ap,test_hits@100
0,Cora,ELPHEdgeAware_log,E1,42,32,64,32,32,0.2,0.010,...,0.596551,0.975574,0.959970,0.631764,0.840758,0.850620,0.730550,0.857617,0.860595,0.665403
1,Cora,ELPHEdgeAware_log,E4,42,64,128,64,64,0.2,0.005,...,0.608705,0.976872,0.967107,0.720509,0.836726,0.851974,0.722960,0.853418,0.861190,0.668246
2,Cora,ELPHEdgeAware_log,E6,42,64,64,128,128,0.3,0.005,...,0.604573,0.974533,0.965697,0.650974,0.832578,0.848934,0.715370,0.850537,0.856479,0.649289
3,Cora,ELPHEdgeAware_log,E3,42,64,64,64,64,0.2,0.010,...,0.523968,0.983219,0.974949,0.785985,0.823864,0.845864,0.709677,0.849828,0.864631,0.665403
4,Cora,ELPHEdgeAware_log,E2,42,64,64,64,64,0.1,0.010,...,0.547503,0.969667,0.953956,0.536255,0.812598,0.832467,0.696395,0.833140,0.837963,0.621801
5,Cora,ELPHEdgeAware_log,E5,42,128,64,64,64,0.2,0.005,...,0.615055,0.949455,0.927072,0.384470,0.820422,0.828590,0.686907,0.829400,0.825818,0.567773


In [11]:
buddy_df_sorted = buddy_df_cora.sort_values(
    by=["val_hits@100", "val_auc", "val_ap"],
    ascending=False,
).reset_index(drop=True)

buddy_df_sorted

,dataset,model,trial_name,seed,predictor_hidden_channels,dropout,lr,best_epoch,best_val_loss,train_auc,train_ap,train_hits@100,val_auc,val_ap,val_hits@100,test_auc,test_ap,test_hits@100
0,Cora,BUDDY_log,B1,42,32,0.0,0.010,8,0.557614,0.992444,0.990951,0.969426,0.884205,0.891664,0.804554,0.903977,0.917355,0.735545
1,Cora,BUDDY_log,B7,42,128,0.3,0.001,11,0.586896,0.986031,0.985120,0.920455,0.861864,0.876795,0.774194,0.882228,0.902464,0.692891
2,Cora,BUDDY_log,B4,42,64,0.3,0.005,3,0.629134,0.984976,0.984307,0.918561,0.860960,0.875762,0.772296,0.880372,0.901606,0.697630
3,Cora,BUDDY_log,B5,42,128,0.1,0.005,2,0.618776,0.983840,0.982753,0.915043,0.863045,0.873251,0.768501,0.880772,0.899607,0.677725
4,Cora,BUDDY_log,B6,42,128,0.2,0.005,3,0.577548,0.985418,0.984256,0.918561,0.862537,0.876325,0.768501,0.881367,0.901750,0.690995
5,Cora,BUDDY_log,B8,42,256,0.2,0.001,7,0.591774,0.985736,0.984236,0.917208,0.861954,0.876028,0.768501,0.880961,0.902162,0.689099
6,Cora,BUDDY_log,B3,42,64,0.2,0.010,7,0.506433,0.993025,0.991401,0.969968,0.863315,0.880909,0.766603,0.891577,0.909968,0.723223
7,Cora,BUDDY_log,B2,42,64,0.1,0.010,7,0.508543,0.993223,0.991632,0.972403,0.860740,0.879937,0.766603,0.889437,0.908784,0.722275


In [12]:
elphedge_summary_cols = [
    "trial_name",
    "hidden_channels",
    "predictor_hidden_channels",
    "message_hidden_channels",
    "update_hidden_channels",
    "dropout",
    "lr",
    "best_val_loss",
    "val_auc",
    "val_ap",
    "val_hits@100",
    "test_auc",
    "test_ap",
    "test_hits@100",
]

buddy_summary_cols = [
    "trial_name",
    "predictor_hidden_channels",
    "dropout",
    "lr",
    "best_val_loss",
    "val_auc",
    "val_ap",
    "val_hits@100",
    "test_auc",
    "test_ap",
    "test_hits@100",
]

print("ELPHEdgeAware_log summary:")
display(elphedge_df_sorted[elphedge_summary_cols])

print("\nBUDDY_log summary:")
display(buddy_df_sorted[buddy_summary_cols])

ELPHEdgeAware_log summary:


,trial_name,hidden_channels,predictor_hidden_channels,message_hidden_channels,update_hidden_channels,dropout,lr,best_val_loss,val_auc,val_ap,val_hits@100,test_auc,test_ap,test_hits@100
0,E1,32,64,32,32,0.2,0.010,0.596551,0.840758,0.850620,0.730550,0.857617,0.860595,0.665403
1,E4,64,128,64,64,0.2,0.005,0.608705,0.836726,0.851974,0.722960,0.853418,0.861190,0.668246
2,E6,64,64,128,128,0.3,0.005,0.604573,0.832578,0.848934,0.715370,0.850537,0.856479,0.649289
3,E3,64,64,64,64,0.2,0.010,0.523968,0.823864,0.845864,0.709677,0.849828,0.864631,0.665403
4,E2,64,64,64,64,0.1,0.010,0.547503,0.812598,0.832467,0.696395,0.833140,0.837963,0.621801
5,E5,128,64,64,64,0.2,0.005,0.615055,0.820422,0.828590,0.686907,0.829400,0.825818,0.567773



BUDDY_log summary:


,trial_name,predictor_hidden_channels,dropout,lr,best_val_loss,val_auc,val_ap,val_hits@100,test_auc,test_ap,test_hits@100
0,B1,32,0.0,0.010,0.557614,0.884205,0.891664,0.804554,0.903977,0.917355,0.735545
1,B7,128,0.3,0.001,0.586896,0.861864,0.876795,0.774194,0.882228,0.902464,0.692891
2,B4,64,0.3,0.005,0.629134,0.860960,0.875762,0.772296,0.880372,0.901606,0.697630
3,B5,128,0.1,0.005,0.618776,0.863045,0.873251,0.768501,0.880772,0.899607,0.677725
4,B6,128,0.2,0.005,0.577548,0.862537,0.876325,0.768501,0.881367,0.901750,0.690995
5,B8,256,0.2,0.001,0.591774,0.861954,0.876028,0.768501,0.880961,0.902162,0.689099
6,B3,64,0.2,0.010,0.506433,0.863315,0.880909,0.766603,0.891577,0.909968,0.723223
7,B2,64,0.1,0.010,0.508543,0.860740,0.879937,0.766603,0.889437,0.908784,0.722275


In [13]:
results_dir = project_root / "results"
results_dir.mkdir(parents=True, exist_ok=True)

elphedge_df_sorted.to_csv(results_dir / "cfg_tuning_elphedge_log_cora.csv", index=False)
buddy_df_sorted.to_csv(results_dir / "cfg_tuning_buddy_log_cora.csv", index=False)

print("Saved tuning results.")

Saved tuning results.


#### Stage2

In [14]:
buddy_trials_cora_stage2 = [
    {"trial_name": "B11", "predictor_hidden_channels": 16, "dropout": 0.0,  "lr": 1e-2},
    {"trial_name": "B12", "predictor_hidden_channels": 32, "dropout": 0.0,  "lr": 1e-2},
    {"trial_name": "B13", "predictor_hidden_channels": 32, "dropout": 0.05, "lr": 1e-2},
    {"trial_name": "B14", "predictor_hidden_channels": 32, "dropout": 0.1,  "lr": 1e-2},
    {"trial_name": "B15", "predictor_hidden_channels": 32, "dropout": 0.0,  "lr": 7e-3},
    {"trial_name": "B16", "predictor_hidden_channels": 64, "dropout": 0.0,  "lr": 1e-2},
]
pd.DataFrame(buddy_trials_cora_stage2)

,trial_name,predictor_hidden_channels,dropout,lr
0,B11,16,0.00,0.010
1,B12,32,0.00,0.010
2,B13,32,0.05,0.010
3,B14,32,0.10,0.010
4,B15,32,0.00,0.007
5,B16,64,0.00,0.010


In [15]:
elphedge_trials_cora_stage2 = [
    {
        "trial_name": "E11",
        "hidden_channels": 32,
        "predictor_hidden_channels": 64,
        "message_hidden_channels": 64,
        "update_hidden_channels": 64,
        "dropout": 0.2,
        "lr": 1e-2,
    },
    {
        "trial_name": "E12",
        "hidden_channels": 32,
        "predictor_hidden_channels": 128,
        "message_hidden_channels": 64,
        "update_hidden_channels": 64,
        "dropout": 0.2,
        "lr": 1e-2,
    },
    {
        "trial_name": "E13",
        "hidden_channels": 64,
        "predictor_hidden_channels": 64,
        "message_hidden_channels": 64,
        "update_hidden_channels": 64,
        "dropout": 0.2,
        "lr": 1e-2,
    },
    {
        "trial_name": "E14",
        "hidden_channels": 64,
        "predictor_hidden_channels": 64,
        "message_hidden_channels": 64,
        "update_hidden_channels": 64,
        "dropout": 0.15,
        "lr": 1e-2,
    },
    {
        "trial_name": "E15",
        "hidden_channels": 64,
        "predictor_hidden_channels": 64,
        "message_hidden_channels": 64,
        "update_hidden_channels": 64,
        "dropout": 0.2,
        "lr": 7e-3,
    },
]

pd.DataFrame(elphedge_trials_cora_stage2)

,trial_name,hidden_channels,predictor_hidden_channels,message_hidden_channels,update_hidden_channels,dropout,lr
0,E11,32,64,64,64,0.20,0.010
1,E12,32,128,64,64,0.20,0.010
2,E13,64,64,64,64,0.20,0.010
3,E14,64,64,64,64,0.15,0.010
4,E15,64,64,64,64,0.20,0.007


In [16]:
elphedge_results_cora_stage2 = []

for cfg in elphedge_trials_cora_stage2:
    print(f"Running Cora {cfg['trial_name']} ...")
    result = run_elphedge_trial(
        dataset_name="Cora",
        train_data_local=train_data,
        val_data_local=val_data,
        test_data_local=test_data,
        **cfg,
    )
    elphedge_results_cora_stage2.append(result)

elphedge_df_cora_stage2 = pd.DataFrame(elphedge_results_cora_stage2).sort_values(
    by=["val_hits@100", "val_auc", "val_ap"],
    ascending=False,
).reset_index(drop=True)

elphedge_df_sorted_stage2 = elphedge_df_cora_stage2.sort_values(
    by=["val_hits@100", "val_auc", "val_ap"],
    ascending=False,
).reset_index(drop=True)

Running Cora E11 ...
Running Cora E12 ...
Running Cora E13 ...
Running Cora E14 ...
Running Cora E15 ...


In [17]:
buddy_results_cora_stage2 = []

for cfg in buddy_trials_cora_stage2:
    print(f"Running Cora {cfg['trial_name']} ...")
    result = run_buddy_trial(
        dataset_name="Cora",
        train_data_local=train_data,
        val_data_local=val_data,
        test_data_local=test_data,
        buddy_cache_local=buddy_cache,
        **cfg,
    )
    buddy_results_cora_stage2.append(result)

buddy_df_cora_stage2 = pd.DataFrame(buddy_results_cora_stage2).sort_values(
    by=["val_hits@100", "val_auc", "val_ap"],
    ascending=False,
).reset_index(drop=True)

buddy_df_sorted_stage2 = buddy_df_cora_stage2.sort_values(
    by=["val_hits@100", "val_auc", "val_ap"],
    ascending=False,
).reset_index(drop=True)

Running Cora B11 ...
Running Cora B12 ...
Running Cora B13 ...
Running Cora B14 ...
Running Cora B15 ...
Running Cora B16 ...


In [18]:
print("ELPHEdgeAware_log summary:")
display(elphedge_df_sorted_stage2[elphedge_summary_cols])

print("\nBUDDY_log summary:")
display(buddy_df_sorted_stage2[buddy_summary_cols])

ELPHEdgeAware_log summary:


,trial_name,hidden_channels,predictor_hidden_channels,message_hidden_channels,update_hidden_channels,dropout,lr,best_val_loss,val_auc,val_ap,val_hits@100,test_auc,test_ap,test_hits@100
0,E12,32,128,64,64,0.20,0.010,0.521902,0.847877,0.861639,0.743833,0.874409,0.881632,0.701422
1,E11,32,64,64,64,0.20,0.010,0.598073,0.840838,0.855864,0.734345,0.861663,0.870008,0.683412
2,E14,64,64,64,64,0.15,0.010,0.574395,0.841140,0.855313,0.722960,0.873857,0.890120,0.697630
3,E13,64,64,64,64,0.20,0.010,0.523970,0.823882,0.845870,0.709677,0.849808,0.864507,0.665403
4,E15,64,64,64,64,0.20,0.007,0.597143,0.820743,0.835159,0.686907,0.839550,0.843018,0.634123



BUDDY_log summary:


,trial_name,predictor_hidden_channels,dropout,lr,best_val_loss,val_auc,val_ap,val_hits@100,test_auc,test_ap,test_hits@100
0,B12,32,0.00,0.010,0.557614,0.884205,0.891664,0.804554,0.903977,0.917355,0.735545
1,B14,32,0.10,0.010,0.555838,0.885567,0.892073,0.802657,0.904561,0.917633,0.734597
2,B13,32,0.05,0.010,0.557669,0.885073,0.892008,0.800759,0.904411,0.917567,0.736493
3,B11,16,0.00,0.010,0.488292,0.893690,0.899683,0.795066,0.912815,0.923460,0.733649
4,B15,32,0.00,0.007,0.522474,0.872277,0.885186,0.785579,0.894061,0.911253,0.710900
5,B16,64,0.00,0.010,0.508196,0.860416,0.879938,0.768501,0.889622,0.908938,0.723223


In [19]:
elphedge_df_sorted_stage2.to_csv(results_dir / "cfg_tuning_elphedge_log_cora_2.csv", index=False)
buddy_df_sorted_stage2.to_csv(results_dir / "cfg_tuning_buddy_log_cora_2.csv", index=False)

print("Saved tuning results.")

Saved tuning results.


### Pubmed

In [21]:
set_seed(42)
device = get_device()

dataset_name = "Pubmed"

data = get_data_object(dataset_name, root=project_root / "data")
train_data, val_data, test_data = prepare_link_prediction_data(dataset_name, data)

print("Device:", device)
print("Dataset:", dataset_name)
print(train_data)
print(val_data)
print(test_data)

Processing...


Device: cuda
Dataset: Pubmed
Data(x=[19717, 500], edge_index=[2, 62056], y=[19717], train_mask=[19717], val_mask=[19717], test_mask=[19717], edge_label=[62056], edge_label_index=[2, 62056])
Data(x=[19717, 500], edge_index=[2, 62056], y=[19717], train_mask=[19717], val_mask=[19717], test_mask=[19717], edge_label=[8864], edge_label_index=[2, 8864])
Data(x=[19717, 500], edge_index=[2, 70920], y=[19717], train_mask=[19717], val_mask=[19717], test_mask=[19717], edge_label=[17728], edge_label_index=[2, 17728])


Done!


In [22]:
buddy_cache = build_buddy_cache(
    x=train_data.x.to(device),
    edge_index=train_data.edge_index.to(device),
    num_hops=2,
    minhash_num_perm=128,
    hll_p=8,
    feature_propagation="mean",
    cache_device=device,
)

print("Pubmed BUDDY cache ready.")
print(buddy_cache.keys())

Pubmed BUDDY cache ready.
dict_keys(['num_nodes', 'num_hops', 'propagated_x_hops', 'minhash_hops', 'hll_hops', 'cardinality_hops'])


In [23]:
elphedge_trials_pubmed = [
    {
        "trial_name": "PE1",
        "hidden_channels": 32,
        "predictor_hidden_channels": 128,
        "message_hidden_channels": 64,
        "update_hidden_channels": 64,
        "dropout": 0.20,
        "lr": 1e-2,
    },
    {
        "trial_name": "PE2",
        "hidden_channels": 32,
        "predictor_hidden_channels": 64,
        "message_hidden_channels": 64,
        "update_hidden_channels": 64,
        "dropout": 0.20,
        "lr": 1e-2,
    },
    {
        "trial_name": "PE3",
        "hidden_channels": 64,
        "predictor_hidden_channels": 128,
        "message_hidden_channels": 64,
        "update_hidden_channels": 64,
        "dropout": 0.20,
        "lr": 5e-3,
    },
    {
        "trial_name": "PE4",
        "hidden_channels": 64,
        "predictor_hidden_channels": 64,
        "message_hidden_channels": 64,
        "update_hidden_channels": 64,
        "dropout": 0.30,
        "lr": 5e-3,
    },
]

pd.DataFrame(elphedge_trials_pubmed)

buddy_trials_pubmed = [
    {"trial_name": "PB1", "predictor_hidden_channels": 16, "dropout": 0.0, "lr": 1e-2},
    {"trial_name": "PB2", "predictor_hidden_channels": 32, "dropout": 0.0, "lr": 1e-2},
    {"trial_name": "PB3", "predictor_hidden_channels": 32, "dropout": 0.1, "lr": 1e-2},
    {"trial_name": "PB4", "predictor_hidden_channels": 32, "dropout": 0.2, "lr": 5e-3},
    {"trial_name": "PB5", "predictor_hidden_channels": 64, "dropout": 0.1, "lr": 5e-3},
]

pd.DataFrame(buddy_trials_pubmed)

,trial_name,predictor_hidden_channels,dropout,lr
0,PB1,16,0.0,0.010
1,PB2,32,0.0,0.010
2,PB3,32,0.1,0.010
3,PB4,32,0.2,0.005
4,PB5,64,0.1,0.005


In [24]:
buddy_results_pubmed = []

for cfg in buddy_trials_pubmed:
    print(f"Running Pubmed {cfg['trial_name']} ...")
    result = run_buddy_trial(
        dataset_name="Pubmed",
        train_data_local=train_data,
        val_data_local=val_data,
        test_data_local=test_data,
        buddy_cache_local=buddy_cache,
        epochs=25,
        patience=8,
        **cfg,
    )
    buddy_results_pubmed.append(result)

buddy_df_pubmed = pd.DataFrame(buddy_results_pubmed).sort_values(
    by=["val_hits@100", "val_auc", "val_ap"],
    ascending=False,
).reset_index(drop=True)

buddy_df_pubmed

Running Pubmed PB1 ...
Running Pubmed PB2 ...
Running Pubmed PB3 ...
Running Pubmed PB4 ...
Running Pubmed PB5 ...


,dataset,model,trial_name,seed,predictor_hidden_channels,dropout,lr,best_epoch,best_val_loss,train_auc,train_ap,train_hits@100,val_auc,val_ap,val_hits@100,test_auc,test_ap,test_hits@100
0,Pubmed,BUDDY_log,PB3,42,32,0.1,0.010,7,0.591955,0.996361,0.995492,0.694534,0.905566,0.914352,0.565659,0.905211,0.914955,0.502595
1,Pubmed,BUDDY_log,PB2,42,32,0.0,0.010,7,0.591902,0.996323,0.995459,0.692310,0.904736,0.913821,0.565433,0.904331,0.914386,0.502820
2,Pubmed,BUDDY_log,PB4,42,32,0.2,0.005,10,0.611816,0.996310,0.995485,0.705395,0.893990,0.906293,0.562951,0.894708,0.907679,0.498872
3,Pubmed,BUDDY_log,PB5,42,64,0.1,0.005,7,0.581960,0.995143,0.993922,0.603616,0.919252,0.923909,0.560018,0.917983,0.923251,0.491539
4,Pubmed,BUDDY_log,PB1,42,16,0.0,0.010,5,0.625671,0.994385,0.993303,0.604809,0.888323,0.901264,0.547157,0.889116,0.902732,0.484206


In [25]:
elphedge_results_pubmed = []

for cfg in elphedge_trials_pubmed:
    print(f"Running Pubmed {cfg['trial_name']} ...")
    result = run_elphedge_trial(
        dataset_name="Pubmed",
        train_data_local=train_data,
        val_data_local=val_data,
        test_data_local=test_data,
        epochs=25,
        patience=8,
        **cfg,
    )
    elphedge_results_pubmed.append(result)

elphedge_df_pubmed = pd.DataFrame(elphedge_results_pubmed).sort_values(
    by=["val_hits@100", "val_auc", "val_ap"],
    ascending=False,
).reset_index(drop=True)

elphedge_df_pubmed

Running Pubmed PE1 ...
Running Pubmed PE2 ...
Running Pubmed PE3 ...
Running Pubmed PE4 ...


,dataset,model,trial_name,seed,hidden_channels,predictor_hidden_channels,message_hidden_channels,update_hidden_channels,dropout,lr,...,best_val_loss,train_auc,train_ap,train_hits@100,val_auc,val_ap,val_hits@100,test_auc,test_ap,test_hits@100
0,Pubmed,ELPHEdgeAware_log,PE3,42,64,128,64,64,0.2,0.005,...,0.701364,0.999343,0.999086,0.983434,0.873212,0.890117,0.543998,0.875877,0.896111,0.515456
1,Pubmed,ELPHEdgeAware_log,PE2,42,32,64,64,64,0.2,0.010,...,0.901631,0.999576,0.999404,0.993844,0.839353,0.874951,0.541968,0.840795,0.881790,0.531250
2,Pubmed,ELPHEdgeAware_log,PE1,42,32,128,64,64,0.2,0.010,...,0.620524,0.993481,0.991661,0.601650,0.856320,0.878995,0.527753,0.889866,0.901558,0.399707
3,Pubmed,ELPHEdgeAware_log,PE4,42,64,64,64,64,0.3,0.005,...,0.836454,0.999104,0.998745,0.958844,0.855629,0.877811,0.505641,0.858220,0.883615,0.470781


In [26]:
print("Pubmed - BUDDY_log")
display(buddy_df_pubmed)

print("Pubmed - ELPHEdgeAware_log")
display(elphedge_df_pubmed)

Pubmed - BUDDY_log


,dataset,model,trial_name,seed,predictor_hidden_channels,dropout,lr,best_epoch,best_val_loss,train_auc,train_ap,train_hits@100,val_auc,val_ap,val_hits@100,test_auc,test_ap,test_hits@100
0,Pubmed,BUDDY_log,PB3,42,32,0.1,0.010,7,0.591955,0.996361,0.995492,0.694534,0.905566,0.914352,0.565659,0.905211,0.914955,0.502595
1,Pubmed,BUDDY_log,PB2,42,32,0.0,0.010,7,0.591902,0.996323,0.995459,0.692310,0.904736,0.913821,0.565433,0.904331,0.914386,0.502820
2,Pubmed,BUDDY_log,PB4,42,32,0.2,0.005,10,0.611816,0.996310,0.995485,0.705395,0.893990,0.906293,0.562951,0.894708,0.907679,0.498872
3,Pubmed,BUDDY_log,PB5,42,64,0.1,0.005,7,0.581960,0.995143,0.993922,0.603616,0.919252,0.923909,0.560018,0.917983,0.923251,0.491539
4,Pubmed,BUDDY_log,PB1,42,16,0.0,0.010,5,0.625671,0.994385,0.993303,0.604809,0.888323,0.901264,0.547157,0.889116,0.902732,0.484206


Pubmed - ELPHEdgeAware_log


,dataset,model,trial_name,seed,hidden_channels,predictor_hidden_channels,message_hidden_channels,update_hidden_channels,dropout,lr,...,best_val_loss,train_auc,train_ap,train_hits@100,val_auc,val_ap,val_hits@100,test_auc,test_ap,test_hits@100
0,Pubmed,ELPHEdgeAware_log,PE3,42,64,128,64,64,0.2,0.005,...,0.701364,0.999343,0.999086,0.983434,0.873212,0.890117,0.543998,0.875877,0.896111,0.515456
1,Pubmed,ELPHEdgeAware_log,PE2,42,32,64,64,64,0.2,0.010,...,0.901631,0.999576,0.999404,0.993844,0.839353,0.874951,0.541968,0.840795,0.881790,0.531250
2,Pubmed,ELPHEdgeAware_log,PE1,42,32,128,64,64,0.2,0.010,...,0.620524,0.993481,0.991661,0.601650,0.856320,0.878995,0.527753,0.889866,0.901558,0.399707
3,Pubmed,ELPHEdgeAware_log,PE4,42,64,64,64,64,0.3,0.005,...,0.836454,0.999104,0.998745,0.958844,0.855629,0.877811,0.505641,0.858220,0.883615,0.470781


In [27]:
results_dir = project_root / "results"
results_dir.mkdir(parents=True, exist_ok=True)

buddy_df_pubmed.to_csv(results_dir / "cfg_tuning_buddy_log_pubmed.csv", index=False)
elphedge_df_pubmed.to_csv(results_dir / "cfg_tuning_elphedge_log_pubmed.csv", index=False)

print("Saved Pubmed tuning results.")

Saved Pubmed tuning results.


### Collab

In [7]:
set_seed(42)
device = get_device()

dataset_name = "Collab"

data = get_data_object(dataset_name, root=project_root / "data")
train_data, val_data, test_data = prepare_link_prediction_data(dataset_name, data)

print("Device:", device)
print("Dataset:", dataset_name)
print(train_data)
print(val_data)
print(test_data)

Device: cuda
Dataset: Collab
Data(num_nodes=235868, edge_index=[2, 1644976], x=[235868, 128], edge_year=[2358104, 1], edge_label=[1644976], edge_label_index=[2, 1644976])
Data(num_nodes=235868, edge_index=[2, 1644976], x=[235868, 128], edge_year=[2358104, 1], edge_label=[96762], edge_label_index=[2, 96762])
Data(num_nodes=235868, edge_index=[2, 1741738], x=[235868, 128], edge_year=[2358104, 1], edge_label=[193526], edge_label_index=[2, 193526])


In [8]:
buddy_cache = build_buddy_cache(
    x=train_data.x.to(device),
    edge_index=train_data.edge_index.to(device),
    num_hops=2,
    minhash_num_perm=128,
    hll_p=8,
    feature_propagation="mean",
    cache_device=device,
)

print("Collab BUDDY cache ready.")
print(buddy_cache.keys())

Collab BUDDY cache ready.
dict_keys(['num_nodes', 'num_hops', 'propagated_x_hops', 'minhash_hops', 'hll_hops', 'cardinality_hops'])


In [9]:
buddy_trials_collab = [
    {"trial_name": "CB1", "predictor_hidden_channels": 32, "dropout": 0.0, "lr": 1e-2},
    {"trial_name": "CB2", "predictor_hidden_channels": 32, "dropout": 0.1, "lr": 1e-2},
    {"trial_name": "CB3", "predictor_hidden_channels": 64, "dropout": 0.1, "lr": 5e-3},
]

pd.DataFrame(buddy_trials_collab)

,trial_name,predictor_hidden_channels,dropout,lr
0,CB1,32,0.0,0.010
1,CB2,32,0.1,0.010
2,CB3,64,0.1,0.005


In [10]:
elphedge_trial_collab = [
    {
    "trial_name": "CE1",
    "hidden_channels": 32,
    "predictor_hidden_channels": 128,
    "message_hidden_channels": 64,
    "update_hidden_channels": 64,
    "dropout": 0.20,
    "lr": 5e-3,
    }
]
pd.DataFrame(elphedge_trial_collab)

,trial_name,hidden_channels,predictor_hidden_channels,message_hidden_channels,update_hidden_channels,dropout,lr
0,CE1,32,128,64,64,0.2,0.005


In [11]:
buddy_results_collab = []

for cfg in buddy_trials_collab:
    print(f"Running {dataset_name} {cfg['trial_name']} ...")
    result = run_buddy_trial(
        dataset_name=dataset_name,
        train_data_local=train_data,
        val_data_local=val_data,
        test_data_local=test_data,
        buddy_cache_local=buddy_cache,
        epochs=25,
        patience=8,
        monitor_hits_k=50,
        **cfg,
    )
    buddy_results_collab.append(result)

    import gc
    gc.collect()
    torch.cuda.empty_cache() # FIXED

buddy_df_collab = pd.DataFrame(buddy_results_collab).sort_values(
    by=["val_hits@50", "val_auc", "val_ap"],
    ascending=False,
).reset_index(drop=True)

buddy_df_collab

Running Collab CB1 ...
Running Collab CB2 ...
Running Collab CB3 ...


,dataset,model,trial_name,seed,predictor_hidden_channels,dropout,lr,best_epoch,best_val_loss,best_monitor_value,train_auc,train_ap,train_hits@50,val_auc,val_ap,val_hits@50,test_auc,test_ap,test_hits@50
0,Collab,BUDDY_log,CB3,42,64,0.1,0.005,25,0.246824,0.852111,0.999793,0.999689,0.248141,0.961541,0.978271,0.852111,0.960995,0.977934,0.693509
1,Collab,BUDDY_log,CB1,42,32,0.0,0.010,25,0.317599,0.848928,0.999797,0.999686,0.252889,0.960209,0.977671,0.848928,0.959560,0.977331,0.690202
2,Collab,BUDDY_log,CB2,42,32,0.1,0.010,25,0.304530,0.848515,0.999794,0.999684,0.240394,0.960171,0.977700,0.848515,0.959548,0.977348,0.688032


In [12]:
elphedge_result_collab = []

for cfg in elphedge_trial_collab:
    print(f"Running {dataset_name} {cfg['trial_name']} ...")

    result = run_elphedge_trial(
        dataset_name=dataset_name,
        train_data_local=train_data,
        val_data_local=val_data,
        test_data_local=test_data,
        epochs=25,
        patience=8,
        monitor_hits_k=50,
        **cfg,
    )
    elphedge_result_collab.append(result)

    import gc
    gc.collect()
    torch.cuda.empty_cache()

elphedge_df_collab = pd.DataFrame(elphedge_result_collab).sort_values(
    by=["val_hits@50", "val_auc", "val_ap"],
    ascending=False,
).reset_index(drop=True)

elphedge_df_collab

Running Collab CE1 ...


KeyboardInterrupt: 

In [13]:
print("Collab - BUDDY_log")
display(buddy_df_collab)

# print("Collab - ELPHEdgeAware_log")
# display(elphedge_df_collab)

Collab - BUDDY_log


,dataset,model,trial_name,seed,predictor_hidden_channels,dropout,lr,best_epoch,best_val_loss,best_monitor_value,train_auc,train_ap,train_hits@50,val_auc,val_ap,val_hits@50,test_auc,test_ap,test_hits@50
0,Collab,BUDDY_log,CB3,42,64,0.1,0.005,25,0.246824,0.852111,0.999793,0.999689,0.248141,0.961541,0.978271,0.852111,0.960995,0.977934,0.693509
1,Collab,BUDDY_log,CB1,42,32,0.0,0.010,25,0.317599,0.848928,0.999797,0.999686,0.252889,0.960209,0.977671,0.848928,0.959560,0.977331,0.690202
2,Collab,BUDDY_log,CB2,42,32,0.1,0.010,25,0.304530,0.848515,0.999794,0.999684,0.240394,0.960171,0.977700,0.848515,0.959548,0.977348,0.688032


In [14]:
results_dir = project_root / "results"
results_dir.mkdir(parents=True, exist_ok=True)

buddy_df_collab.to_csv(results_dir / "cfg_tuning_buddy_log_collab.csv", index=False)
# elphedge_df_collab.to_csv(results_dir / "cfg_tuning_elphedge_log_collab.csv", index=False)

print("Saved Collab tuning results.")

Saved Collab tuning results.
